In [ ]:
!pip install -U transformers accelerate peft bitsandbytes

!pip install qwen-vl-utils

!pip uninstall -y torchao

!pip install torchao>=0.16.0

!pip install fastapi uvicorn pyngrok python-multipart

In [ ]:
import io
import re
import json
import torch

from PIL import Image

from fastapi import (
    FastAPI,
    UploadFile,
    Form
)

from fastapi.middleware.cors import CORSMiddleware

from pyngrok import ngrok

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor
)

from qwen_vl_utils import process_vision_info

from peft import PeftModel

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

LORA_PATH = "imbee510/qwen2-5-vl-landmark-vietnam-lora"


# =========================================================
# LOAD BASE MODEL
# =========================================================

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(

    BASE_MODEL,

    torch_dtype=torch.float16,

    device_map="auto"
)


# =========================================================
# LOAD LORA
# =========================================================

model = PeftModel.from_pretrained(

    base_model,

    LORA_PATH
)


# =========================================================
# LOAD PROCESSOR
# =========================================================

processor = AutoProcessor.from_pretrained(
    BASE_MODEL
)

print("Model loaded successfully.")

In [ ]:
SYSTEM_PROMPT = """
You are an expert Vietnamese cultural heritage
and landmark recognition AI.

Your task is to analyze the provided image carefully
using ONLY visible visual evidence.

You must identify:

- landmark name
- city
- country
- visual reasoning
- uncertainty level

=========================================================
CRITICAL RULES
=========================================================

1. DO NOT hallucinate landmarks.

2. PRIVACY & DOMAIN FILTERING (CRITICAL):
- If the image does NOT contain a landscape, historical site, monument, or recognized geographical landmark (e.g., if the image is a person's face, a selfie, an animal, an indoor object, a document, or anything unrelated), you MUST refuse to identify it.
- In this case, you MUST set `landmark_name` to exactly "OUT_OF_DOMAIN" and explain the reason in `vision_reasoning` (e.g. "This image contains personal identifiable information or is unrelated to landmarks.").
- Do NOT guess or return any alternative candidates for out-of-domain images.

3. ONLY identify landmarks that are visually supported
by the image.

3. If uncertain:
- lower confidence
- explain ambiguity
- avoid guessing aggressively

4. Avoid overconfidence.

5. Use confidence above 0.9 ONLY if the landmark
is extremely recognizable.

6. Use visual evidence such as:
- architectural structure
- clock towers
- roofs
- bridges
- statues
- rivers
- surrounding environment
- signs or text
- cultural symbols

7. If multiple landmarks are possible,
mention uncertainty in reasoning.

8. If the landmark cannot be confidently identified,
return "Unknown".

=========================================================
ALTERNATIVE CANDIDATES RULES
=========================================================

- If the landmark is ambiguous or visually similar
to multiple locations, provide possible alternatives.

- Include up to 3 possible landmark candidates.

- Candidates should ONLY be visually plausible.

- If confidence is high (> 0.90),
alternative_candidates should usually be empty.

- If uncertainty exists, include reasonable alternatives.

- Do NOT invent random landmarks.

- If no reasonable alternatives exist,
return an empty list.

=========================================================
OUTPUT FORMAT
=========================================================

Return valid JSON ONLY.

{
    "landmark_name": "...",

    "city": "...",

    "country": "...",

    "vision_reasoning": "...",

    "alternative_candidates": []
}

=========================================================
IMPORTANT
=========================================================

Return JSON ONLY.

Do not generate markdown.

Do not explain outside JSON.
"""

In [ ]:
class VisionModel:

    def __init__(self):

        self.model = model

        self.processor = processor

    # =====================================================
    # MAIN PREDICTION
    # =====================================================

    def predict(
        self,
        prompt,
        image_path
    ):

        # -------------------------------------------------
        # BUILD MESSAGES
        # -------------------------------------------------

        messages = [

            {
                "role": "system",

                "content": SYSTEM_PROMPT
            },

            {
                "role": "user",

                "content": [

                    {
                        "type": "image",

                        "image": image_path
                    },

                    {
                        "type": "text",

                        "text": prompt
                    }
                ]
            }
        ]

        # -------------------------------------------------
        # APPLY CHAT TEMPLATE
        # -------------------------------------------------

        text = self.processor.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True
        )

        # -------------------------------------------------
        # PROCESS IMAGE
        # -------------------------------------------------

        image_inputs, video_inputs = process_vision_info(
            messages
        )

        # -------------------------------------------------
        # TOKENIZE
        # -------------------------------------------------

        inputs = self.processor(

            text=[text],

            images=image_inputs,

            videos=video_inputs,

            padding=True,

            return_tensors="pt"
        )

        inputs = inputs.to("cuda")

        # -------------------------------------------------
        # GENERATE
        # -------------------------------------------------

        outputs = self.model.generate(

            **inputs,

            max_new_tokens=512,

            temperature=0.01,

            output_scores=True,

            return_dict_in_generate=True
        )

        generated_ids = outputs.sequences

        import torch
        import torch.nn.functional as F

        try:
            input_len = inputs.input_ids.shape[1]
        except AttributeError:
            input_len = inputs["input_ids"].shape[1]

        gen_logits = torch.stack(outputs.scores, dim=1)
        gen_probs = F.softmax(gen_logits, dim=-1)
        generated_tokens = generated_ids[:, input_len:]
        token_probs = torch.gather(gen_probs, 2, generated_tokens.unsqueeze(-1)).squeeze(-1)
        real_confidence = token_probs.min().item()

        # -------------------------------------------------
        # REMOVE INPUT TOKENS
        # -------------------------------------------------

        generated_ids_trimmed = [

            out_ids[len(in_ids):]

            for in_ids, out_ids in zip(
                inputs.input_ids,
                generated_ids
            )
        ]

        # -------------------------------------------------
        # DECODE
        # -------------------------------------------------

        output_text = self.processor.batch_decode(

            generated_ids_trimmed,

            skip_special_tokens=True,

            clean_up_tokenization_spaces=False
        )[0]

        #real_confidence = 0.95

        return output_text, real_confidence

In [ ]:
vision_model = VisionModel()

In [ ]:
app = FastAPI()


# =========================================================
# CORS
# =========================================================

app.add_middleware(

    CORSMiddleware,

    allow_origins=["*"],

    allow_credentials=True,

    allow_methods=["*"],

    allow_headers=["*"],
)

In [ ]:
@app.post("/predict_landmark")
async def detect(

    file: UploadFile,

    prompt: str = Form(...)
):

    # =====================================================
    # READ IMAGE
    # =====================================================

    image_data = await file.read()

    image = Image.open(
        io.BytesIO(image_data)
    )

    # =====================================================
    # RUN MODEL
    # =====================================================

    result_text, real_confidence = vision_model.predict(

        prompt=prompt,

        image_path=image
    )

    print("\nMODEL OUTPUT:\n")
    print(result_text)

    # =====================================================
    # TRY PARSE JSON
    # =====================================================

    try:

        cleaned = re.sub(
            r"```json|```",
            "",
            result_text
        ).strip()

        parsed = json.loads(cleaned)

        parsed["reasoning_confidence"] = real_confidence

        return parsed

    # =====================================================
    # FALLBACK
    # =====================================================

    except Exception:

        return {

            "landmark_name": "Unknown",

            "city": "Unknown",

            "country": "Unknown",

            "reasoning_confidence": 0.0,

            "vision_reasoning":
            "Failed to parse model output.",

            "alternative_candidates": [],

            "raw_output": result_text
        }

In [ ]:
!pip install nest_asyncio

In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
from pyngrok import ngrok
import uvicorn
import asyncio
import threading
import os
from dotenv import load_dotenv

load_dotenv()

# =========================================================
# AUTH TOKEN
# =========================================================
ngrok_token = os.getenv('NGROK_TOKEN')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print("Caution: Can not fine NGROK_TOKEN in .env")

# =========================================================
# START NGROK
# =========================================================
public_url = ngrok.connect(8001)
print('='*60)
print(f'YOUR NGROK URL: {public_url.public_url}')
print('='*60)

# =========================================================
# START UVICORN SERVER (Run safely in background thread)
# =========================================================
def run_server():
    # Create a fresh asyncio loop for this specific thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(app, host='0.0.0.0', port=8001)
    server = uvicorn.Server(config)

    # Start the server using the thread\'s loop
    loop.run_until_complete(server.serve())

# Run in background so Colab doesn\'t block
thread = threading.Thread(target=run_server)
thread.start()
print('✅ Server is running in the background!')
